# Notebook 03 - Train model du doan gia nha (Ha Noi)

Muc tieu:
- Train baseline + model chinh tren du lieu clean.
- Danh gia theo MAE/RMSE/R2.
- Luu model tot nhat vao `webDemo/BackEnd/model1.joblib`.

In [2]:
%pip install numpy pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from joblib import dump

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "housing_hanoi_clean.csv"
MODEL_PATH = PROJECT_ROOT / "webDemo" / "BackEnd" / "model1.joblib"

DATA_PATH, MODEL_PATH

(WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/processed/housing_hanoi_clean.csv'),
 WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/webDemo/BackEnd/model1.joblib'))

In [4]:
df = pd.read_csv(DATA_PATH)
print("Shape du lieu clean (tu Notebook 02):", df.shape)
df.head(3)

Shape du lieu clean (tu Notebook 02): (260, 14)


,Gia,Gia_m2,dienTich,chieuNgang,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,list_id
0,9.000000e+09,2.571429e+08,35.0,3.5,10.0,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Khong ro,Phường Ngọc Lâm,Quận Long Biên,129628407
1,1.760000e+10,3.384615e+08,52.0,4.4,12.0,4.0,4.0,6.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất cao cấp,Phường Long Biên,Quận Long Biên,130919547
2,6.500000e+09,2.124183e+08,30.6,7.0,4.3,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Nội thất đầy đủ,Phường Long Biên,Quận Long Biên,130571107


## 1) Chon feature dung theo backend

In [5]:
# Phai khop PredictorService.FEATURE_ORDER trong backend
FEATURES = [
    "chieuDai",
    "chieuNgang",
    "dienTich",
    "Phongngu",
    "SoTang",
    "PhongTam",
    "Loai",
    "GiayTo",
    "TinhTrangNoiThat",
    "Phuong",
]
TARGET = "Gia"

missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Thieu cot feature: {missing_features}")

X = df[FEATURES].copy()
y = pd.to_numeric(df[TARGET], errors="coerce")

valid_idx = y.notna()
X = X[valid_idx]
y = y[valid_idx]

# Log-target de giam anh huong outlier gia
y_log = np.log1p(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("y_log shape:", y_log.shape)

X shape: (260, 10)
y shape: (260,)
y_log shape: (260,)


In [6]:
numeric_features = ["chieuDai", "chieuNgang", "dienTich", "Phongngu", "SoTang", "PhongTam"]
categorical_features = ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong"]

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 2) Train/Test split + train RandomForest

In [7]:
X_train, X_test, y_train, y_test, y_train_log, y_test_log = train_test_split(
    X, y, y_log, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", rf_model)])
pipeline.fit(X_train, y_train_log)

# Du doan tren scale log, sau do doi ve scale gia goc
pred_log = pipeline.predict(X_test)
pred = np.expm1(pred_log)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

# CV_RMSE tren scale log de theo doi on dinh huan luyen
cv_neg_rmse_log = cross_val_score(
    pipeline,
    X_train,
    y_train_log,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
cv_rmse_log_mean = -cv_neg_rmse_log.mean()

metrics_df = pd.DataFrame([
    {
        "model": "RandomForest_log_target",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_RMSE_LOG": cv_rmse_log_mean,
    }
])
metrics_df

,model,MAE,RMSE,R2,CV_RMSE_LOG
0,RandomForest_log_target,3.846866e+09,5.099814e+09,0.820099,0.323349


## 3) Luu model RandomForest vao backend

In [8]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(pipeline, MODEL_PATH)

print("Model su dung: RandomForest")
print("Da luu model tai:", MODEL_PATH)

Model su dung: RandomForest
Da luu model tai: C:\Users\asus\Desktop\DuDoanGiaNha\webDemo\BackEnd\model1.joblib


In [13]:
# Test nhanh bang 1 mau co dinh de so sanh nhat quan giua cac lan train
fixed_sample = pd.DataFrame([
    {
  "chieuDai": 12.0,
  "chieuNgang": 5,
  "dienTich": 45.0,
  "Phongngu": 3.0,
  "SoTang": 4.0,
  "PhongTam": 5.0,
  "Loai": "Nhà ngõ, hẻm",
  "GiayTo": "Khong ro",
  "TinhTrangNoiThat": "Khong ro",
  "Phuong": "Phường Long Biên"
}
])

pred_raw = pipeline.predict(fixed_sample)[0]
if "log_target" in str(metrics_df.loc[0, "model"]).lower():
    pred_vnd = float(np.expm1(pred_raw))
else:
    pred_vnd = float(pred_raw)

print("Du doan mau co dinh (VND):", pred_vnd)

Du doan mau co dinh (VND): 9262776891.161615
